# Recipe Override with ModelTrainer.from_recipe

This notebook demonstrates the **`get_resolved_recipe()`** feature on `ModelTrainer.from_recipe`.

With `ModelTrainer`, you provide a recipe YAML file and optional `recipe_overrides` dict.
The new `get_resolved_recipe()` method lets you inspect the fully merged configuration
before submitting the training job.

In [1]:
# Setup environment
import sys
import os
sys.path.insert(0, '../../sagemaker-train/src')
sys.path.insert(0, '../../sagemaker-core/src')

# Point botocore at the bundled service model
os.environ['AWS_DATA_PATH'] = os.path.abspath('../../sagemaker-core/sample')

!pip install -q omegaconf pyyaml

/Users/nargokul/.zshenv:1: command not found: 1805912728

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 1: Create a Nova Recipe YAML File

A recipe file for `ModelTrainer` follows the Nova recipe structure with sections like
`run`, `training_config`, etc.

In [2]:
import yaml

# Define a Nova-style training recipe
recipe_config = {
    "run": {
        "name": "my-nova-sft-experiment",
        "model_type": "amazon.nova-lite",
        "model_name_or_path": "amazon-nova-lite-v2",
        "replicas": 1,
    },
    "training_config": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 4,
        "sequence_length": 4096,
        "warmup_steps": 100,
    },
}

recipe_path = "my_nova_recipe.yaml"
with open(recipe_path, "w") as f:
    yaml.dump(recipe_config, f, default_flow_style=False)

print(f"Recipe written to: {os.path.abspath(recipe_path)}")
print("\nContents:")
with open(recipe_path) as f:
    print(f.read())

Recipe written to: /Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/my_nova_recipe.yaml

Contents:
run:
  model_name_or_path: amazon-nova-lite-v2
  model_type: amazon.nova-lite
  name: my-nova-sft-experiment
  replicas: 1
training_config:
  batch_size: 4
  learning_rate: 1.0e-05
  num_epochs: 3
  sequence_length: 4096
  warmup_steps: 100



## Step 2: Create ModelTrainer with Recipe + Overrides

`ModelTrainer.from_recipe` accepts:
- `training_recipe`: path to a local YAML file or a recipe name from the HyperPod recipes repo
- `recipe_overrides`: a nested dict that overrides values in the recipe (highest precedence)

In [3]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute

compute = Compute(instance_type="ml.p5.48xlarge", instance_count=1)

model_trainer = ModelTrainer.from_recipe(
    training_recipe="./my_nova_recipe.yaml",
    compute=compute,
    training_image="123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest",
    # Override specific values for this run
    recipe_overrides={
        "run": {"name": "experiment-42"},
        "training_config": {
            "learning_rate": 2e-5,
            "num_epochs": 5,
        },
    },
)

print("ModelTrainer created from recipe with overrides.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:7865: SyntaxWarning: invalid escape sequence '\|'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:8683: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:9039: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:905

[06/09/26 09:00:52] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=89140;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5137;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/nargokul/Library/Application Support/sagemaker/config.yaml


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=414451;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=293021;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/09/26 09:00:53] INFO     SageMaker session not provided. Using default Session.                  ]8;id=947582;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=322629;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

...

                    INFO     Base name not provided. Using default name:                             ]8;id=480556;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=272587;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#90\90]8;;\
                             nova-training-job                                                                     

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=606436;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=700447;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[06/09/26 09:00:54] INFO     OutputDataConfig not provided. Using default:                          ]8;id=174728;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=625509;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-211125564141/nova-training-jo                
                             b' kms_key_id=None compression_type='GZIP'                                            

                    INFO     Training image URI:                                               ]8;id=89911;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=409274;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest                     

ModelTrainer created from recipe with overrides.


## Step 3: Inspect with `get_resolved_recipe()`

Before spending compute, see the exact config that will be used.
This merges: base recipe YAML + recipe_overrides (overrides win).

In [4]:
# See the fully resolved recipe
resolved = model_trainer.get_resolved_recipe()

print("Resolved recipe configuration:")
print("=" * 50)
for section, params in resolved.items():
    print(f"\n[{section}]")
    if isinstance(params, dict):
        for key, value in params.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {params}")

Resolved recipe configuration:

[run]
  model_name_or_path: amazon-nova-lite-v2
  model_type: amazon.nova-lite
  name: experiment-42
  replicas: 1

[training_config]
  batch_size: 4
  learning_rate: 2e-05
  num_epochs: 5
  sequence_length: 4096
  warmup_steps: 100


## Step 4: Verify Precedence

| Parameter | Recipe File | Override | Final Value | Source |
|-----------|------------|----------|-------------|--------|
| `run.name` | my-nova-sft-experiment | experiment-42 | **experiment-42** | Override |
| `training_config.learning_rate` | 1e-5 | 2e-5 | **2e-5** | Override |
| `training_config.num_epochs` | 3 | 5 | **5** | Override |
| `training_config.batch_size` | 4 | (not set) | **4** | Recipe file |
| `training_config.sequence_length` | 4096 | (not set) | **4096** | Recipe file |

In [5]:
# Verify precedence
resolved = model_trainer.get_resolved_recipe()

# Overrides win
assert resolved["run"]["name"] == "experiment-42", "Override should win"
assert resolved["training_config"]["learning_rate"] == 2e-5, "Override should win"
assert resolved["training_config"]["num_epochs"] == 5, "Override should win"

# Recipe file values preserved where no override
assert resolved["training_config"]["batch_size"] == 4, "Recipe value preserved"
assert resolved["training_config"]["sequence_length"] == 4096, "Recipe value preserved"
assert resolved["training_config"]["warmup_steps"] == 100, "Recipe value preserved"

print("All precedence checks passed!")
print(f"\n  run.name: {resolved['run']['name']} (overridden)")
print(f"  learning_rate: {resolved['training_config']['learning_rate']} (overridden)")
print(f"  num_epochs: {resolved['training_config']['num_epochs']} (overridden)")
print(f"  batch_size: {resolved['training_config']['batch_size']} (from recipe)")
print(f"  sequence_length: {resolved['training_config']['sequence_length']} (from recipe)")
print(f"  warmup_steps: {resolved['training_config']['warmup_steps']} (from recipe)")

All precedence checks passed!

  run.name: experiment-42 (overridden)
  learning_rate: 2e-05 (overridden)
  num_epochs: 5 (overridden)
  batch_size: 4 (from recipe)
  sequence_length: 4096 (from recipe)
  warmup_steps: 100 (from recipe)


## Step 5: Iterate — Fix and Re-inspect

If something looks wrong in the resolved recipe, create a new trainer with corrected overrides
and inspect again before submitting.

In [6]:
# Suppose we realize learning_rate is too high — create a new trainer with corrected value
model_trainer_v2 = ModelTrainer.from_recipe(
    training_recipe="./my_nova_recipe.yaml",
    compute=compute,
    training_image="123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest",
    recipe_overrides={
        "run": {"name": "experiment-42-v2"},
        "training_config": {
            "learning_rate": 1e-5,  # Corrected: back to recipe default
            "num_epochs": 5,
        },
    },
)

resolved_v2 = model_trainer_v2.get_resolved_recipe()
print(f"v2 learning_rate: {resolved_v2['training_config']['learning_rate']}")
print(f"v2 run.name: {resolved_v2['run']['name']}")
print("\nReady to submit when satisfied.")

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=236467;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=772019;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/09/26 09:00:55] INFO     SageMaker session not provided. Using default Session.                  ]8;id=965121;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=925884;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Role not provided. Using default role:                                  ]8;id=797337;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=895480;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::211125564141:role/Admin                                                  

                    INFO     Base name not provided. Using default name:                             ]8;id=935443;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=198593;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#90\90]8;;\
                             nova-training-job                                                                     

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=279692;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=662611;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[06/09/26 09:00:56] INFO     OutputDataConfig not provided. Using default:                          ]8;id=409669;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=795108;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-211125564141/nova-training-jo                
                             b' kms_key_id=None compression_type='GZIP'                                            

                    INFO     Training image URI:                                               ]8;id=422997;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=105965;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest                     

v2 learning_rate: 1e-05
v2 run.name: experiment-42-v2

Ready to submit when satisfied.


## Step 6: Nested Recipe Attributes

Real-world recipes often contain **nested structures** — for example, an `lr_scheduler`
sub-dict inside `training_config`. When these are passed to the SageMaker Training API,
nested dicts are automatically **deep-flattened** so that only scalar leaf values become
hyperparameters.

```yaml
training_config:
  learning_rate: 1e-5
  lr_scheduler:          # <-- nested dict
    warmup_steps: 15
    min_lr: 1e-6
  peft:                  # <-- another nested dict
    peft_type: "lora"
    rank: 16
    alpha: 32
```

After flattening, the training job receives:
```
learning_rate = "1e-05"
warmup_steps  = "15"
min_lr        = "1e-06"
peft_type     = "lora"
rank          = "16"
alpha         = "32"
```

You can override any nested key by mirroring the structure in `recipe_overrides`.

In [7]:
import yaml

# Define a recipe with nested attributes
nested_recipe_config = {
    "run": {
        "name": "nested-recipe-demo",
        "model_type": "amazon.nova-lite",
        "model_name_or_path": "amazon-nova-lite-v2",
        "replicas": 1,
    },
    "training_config": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 4,
        "sequence_length": 4096,
        "lr_scheduler": {
            "warmup_steps": 15,
            "min_lr": 1e-6,
        },
        "peft": {
            "peft_type": "lora",
            "rank": 16,
            "alpha": 32,
            "dropout": 0.05,
        },
    },
}

nested_recipe_path = "my_nested_recipe.yaml"
with open(nested_recipe_path, "w") as f:
    yaml.dump(nested_recipe_config, f, default_flow_style=False)

print("Recipe with nested attributes:")
print("=" * 50)
with open(nested_recipe_path) as f:
    print(f.read())

Recipe with nested attributes:
run:
  model_name_or_path: amazon-nova-lite-v2
  model_type: amazon.nova-lite
  name: nested-recipe-demo
  replicas: 1
training_config:
  batch_size: 4
  learning_rate: 1.0e-05
  lr_scheduler:
    min_lr: 1.0e-06
    warmup_steps: 15
  num_epochs: 3
  peft:
    alpha: 32
    dropout: 0.05
    peft_type: lora
    rank: 16
  sequence_length: 4096



In [8]:
# Create a ModelTrainer with nested overrides
nested_trainer = ModelTrainer.from_recipe(
    training_recipe="./my_nested_recipe.yaml",
    compute=compute,
    training_image="123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest",
    # Override a deeply nested key
    recipe_overrides={
        "training_config": {
            "lr_scheduler": {"warmup_steps": 30},  # override nested value
            "peft": {"rank": 32, "alpha": 64},     # override multiple nested values
        },
    },
)

resolved = nested_trainer.get_resolved_recipe()

print("Resolved recipe with nested overrides:")
print("=" * 50)
for section, params in resolved.items():
    print(f"\n[{section}]")
    if isinstance(params, dict):
        for key, value in params.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    {k}: {v}")
            else:
                print(f"  {key}: {value}")
    else:
        print(f"  {params}")

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=798585;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=216415;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/09/26 09:00:57] INFO     SageMaker session not provided. Using default Session.                  ]8;id=281174;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=464978;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Role not provided. Using default role:                                  ]8;id=852167;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=762342;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::211125564141:role/Admin                                                  

                    INFO     Base name not provided. Using default name:                             ]8;id=586655;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=89745;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#90\90]8;;\
                             nova-training-job                                                                     

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=36118;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=791212;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[06/09/26 09:00:58] INFO     OutputDataConfig not provided. Using default:                          ]8;id=384016;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=825776;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-211125564141/nova-training-jo                
                             b' kms_key_id=None compression_type='GZIP'                                            

                    INFO     Training image URI:                                               ]8;id=208915;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=85140;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/training-examples/../../sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             123456789012.dkr.ecr.us-west-2.amazonaws.com/nova-training:latest                     

Resolved recipe with nested overrides:

[run]
  model_name_or_path: amazon-nova-lite-v2
  model_type: amazon.nova-lite
  name: nested-recipe-demo
  replicas: 1

[training_config]
  batch_size: 4
  learning_rate: 1e-05
  lr_scheduler:
    min_lr: 1e-06
    warmup_steps: 30
  num_epochs: 3
  peft:
    alpha: 64
    dropout: 0.05
    peft_type: lora
    rank: 32
  sequence_length: 4096


In [9]:
# Demonstrate flattening: nested dicts become flat hyperparameters
from sagemaker.train.recipe_resolver import flatten_resolved_recipe

flat_hp = flatten_resolved_recipe(resolved)

print("Flattened hyperparameters (what the training job receives):")
print("=" * 50)
for key, value in sorted(flat_hp.items()):
    print(f"  {key}: {value}")

print("\n\nVerifying nested overrides took effect:")
assert flat_hp["warmup_steps"] == 30, "Nested override should win"
assert flat_hp["rank"] == 32, "Nested override should win"
assert flat_hp["alpha"] == 64, "Nested override should win"

print("  warmup_steps = 30  (overridden from 15)")
print("  rank = 32          (overridden from 16)")
print("  alpha = 64         (overridden from 32)")

print("\nVerifying non-overridden nested values preserved:")
assert flat_hp["min_lr"] == 1e-6, "Non-overridden nested value preserved"
assert flat_hp["peft_type"] == "lora", "Non-overridden nested value preserved"
assert flat_hp["dropout"] == 0.05, "Non-overridden nested value preserved"

print("  min_lr = 1e-6      (from recipe, not overridden)")
print("  peft_type = lora   (from recipe, not overridden)")
print("  dropout = 0.05     (from recipe, not overridden)")

print("\nAll nested attribute checks passed!")

Flattened hyperparameters (what the training job receives):
  alpha: 64
  batch_size: 4
  dropout: 0.05
  learning_rate: 1e-05
  min_lr: 1e-06
  model_name_or_path: amazon-nova-lite-v2
  model_type: amazon.nova-lite
  name: nested-recipe-demo
  num_epochs: 3
  peft_type: lora
  rank: 32
  replicas: 1
  sequence_length: 4096
  warmup_steps: 30


Verifying nested overrides took effect:
  warmup_steps = 30  (overridden from 15)
  rank = 32          (overridden from 16)
  alpha = 64         (overridden from 32)

Verifying non-overridden nested values preserved:
  min_lr = 1e-6      (from recipe, not overridden)
  peft_type = lora   (from recipe, not overridden)
  dropout = 0.05     (from recipe, not overridden)

All nested attribute checks passed!


### Nested Override Precedence Table

| Parameter | Recipe File | Override | Final (Flat) | Source |
|-----------|------------|----------|--------------|--------|
| `training_config.learning_rate` | 1e-5 | (not set) | **1e-05** | Recipe |
| `training_config.lr_scheduler.warmup_steps` | 15 | 30 | **30** | Override |
| `training_config.lr_scheduler.min_lr` | 1e-6 | (not set) | **1e-06** | Recipe |
| `training_config.peft.peft_type` | lora | (not set) | **lora** | Recipe |
| `training_config.peft.rank` | 16 | 32 | **32** | Override |
| `training_config.peft.alpha` | 32 | 64 | **64** | Override |
| `training_config.peft.dropout` | 0.05 | (not set) | **0.05** | Recipe |

**Key behavior:** Nested dicts are recursively walked — only scalar leaf values are extracted.
Dict and list values are never passed as hyperparameters directly.

## Step 7: Submit Training Job

Once the resolved config looks correct, submit the job.

In [10]:
# Submit training (requires valid training_image, role, and data setup)
# model_trainer.train(wait=False)

print("To submit training, uncomment model_trainer.train() above.")
print("Requires: valid training image URI, IAM role, and compute resources.")

To submit training, uncomment model_trainer.train() above.
Requires: valid training image URI, IAM role, and compute resources.


## Summary

`ModelTrainer.from_recipe` + `get_resolved_recipe()` enables:

- **Inspect before submit** — See the exact merged config without launching a job
- **Iterate quickly** — Adjust overrides and re-inspect in a notebook loop
- **Nested attribute support** — Override deeply nested keys (e.g., `lr_scheduler.warmup_steps`) and they auto-flatten into hyperparameters
- **Debug failures** — After a failed job, check what was actually submitted
- **Audit trail** — Log the resolved config alongside the training job for reproducibility

**Resolution order:** base recipe YAML → `recipe_overrides` dict (wins)

**Nested keys:** Dicts at any depth are recursively walked; only scalar leaf values become hyperparameters. List and dict values are never passed directly to the training API.